# Simulação Política com Organismos Digitais — Notebook de Cenários
Implementa o spec de simulação eleitoral (seções 6–16) com **engine heurístico substituto do Laya**.

O `HeuristicLayaEngine` segue exatamente o *fallback determinístico* da seção 15.5 do spec
(score = memória + bem-estar ponderado + compatibilidade ideológica + bônus de promessa crível)
e simula `answer_confidence` / `probabilities` no contrato da seção 19.1.

> **Para produção:** troque pela classe `RealLayaEngine` (célula final), que usa `pip install laya`.
> A interface `DecisionEngine` é idêntica — o resto do notebook não muda.

In [1]:
# ============================================================
# CÉLULA 2 — imports e seed
# ============================================================
import math, random, json
from dataclasses import dataclass, field
from numpy.random import default_rng

SEED = 42
rng = default_rng(SEED)
random.seed(SEED)

In [2]:
# ============================================================# CÉLULA 3 — modelo de dados: memória, cidadão, candidato
# ============================================================
@dataclass
class Memory:
    tick: int
    government_id: str          # "government_A" | "government_B" | "government_C" | None
    event_type: str
    personal_impact: float      # -10 a +10
    emotional_weight: float
    perceived_responsibility: float
    decay_half_life_ticks: float = 24.0
    tags: list = field(default_factory=list)

    def effective_impact(self, current_tick: int) -> float:
        age = max(0, current_tick - self.tick)
        recency = 0.5 ** (age / self.decay_half_life_ticks)
        return self.personal_impact * self.emotional_weight * self.perceived_responsibility * recency


VALUES_DIMS = ["seguranca_ordem", "igualdade", "liberdade", "crescimento"]


@dataclass
class Citizen:
    citizen_id: str
    age: int
    region: str
    income_bracket: str          # low | middle | high
    employment_status: str
    health_condition: str
    household_children: int
    groups: list
    political_values: dict       # vetor alinhado a VALUES_DIMS (0..1)
    institutional_trust: float = 50.0
    political_engagement: float = 0.8
    wellbeing: dict = field(default_factory=dict)  # economic, health, safety, education, freedom, environment (0..100)
    memory: list = field(default_factory=list)
    incumbent: str = None        # candidate_id do governo atual

    def memory_by_government(self, current_tick: int) -> dict:
        out = {}
        for m in self.memory:
            if m.government_id:
                out[m.government_id] = out.get(m.government_id, 0.0) + m.effective_impact(current_tick)
        return out

    def add_memory(self, mem: Memory, limit: int = 50, current_tick: int = 0):
        self.memory.append(mem)
        if len(self.memory) > limit:
            protected = {"corruption_scandal", "promise_broken"}
            ranked = sorted(self.memory, key=lambda m: (m.event_type in protected,
                                                        m.effective_impact(current_tick) < 0,
                                                        abs(m.effective_impact(current_tick))), reverse=True)
            self.memory = ranked[:limit]


@dataclass
class Candidate:
    candidate_id: str
    name: str
    ideology_label: str
    ideology_vector: dict        # alinhado a VALUES_DIMS
    administrative_competence: float
    promise_reliability: float
    corruption_risk: float
    communication_skill: float
    target_groups: list
    incumbency_penalty: float = 0.0

In [3]:
# ============================================================# CÉLULA 4 — candidatos A, B, C (seções 9–11 do spec)
# ============================================================
CANDIDATES = {
    "candidate_A": Candidate(
        "candidate_A", "Ordem Nacional", "ordem",
        {"seguranca_ordem": 0.95, "igualdade": 0.25, "liberdade": 0.30, "crescimento": 0.55},
        administrative_competence=0.7, promise_reliability=0.75,
        corruption_risk=0.35, communication_skill=0.8,
        target_groups=["high_income", "business_owner"]),
    "candidate_B": Candidate(
        "candidate_B", "Bem-Estar Coletivo", "bem_estar",
        {"seguranca_ordem": 0.30, "igualdade": 0.95, "liberdade": 0.65, "crescimento": 0.35},
        administrative_competence=0.65, promise_reliability=0.7,
        corruption_risk=0.25, communication_skill=0.7,
        target_groups=["low_income", "elderly", "family_with_children", "worker"]),
    "candidate_C": Candidate(
        "candidate_C", "Crescimento Prático", "crescimento",
        {"seguranca_ordem": 0.45, "igualdade": 0.30, "liberdade": 0.55, "crescimento": 0.95},
        administrative_competence=0.8, promise_reliability=0.65,
        corruption_risk=0.45, communication_skill=0.75,
        target_groups=["business_owner", "rural_producer", "high_income"]),
}

In [4]:
# ============================================================# CÉLULA 5 — motor de decisão (interface + heurístico)
# ============================================================
# O HeuristicLayaEngine implementa exatamente o fallback determinístico
# da seção 15.5 do spec, e *simula* answer_confidence/probabilities.
# Para produção, troque por RealLayaEngine (célula final).

WELLBEING_TO_PRIORITY = {
    "safety": "safety", "health": "health", "education": "education",
    "economic": "cost_of_living", "environment": "environment", "freedom": "freedom",
}


class DecisionEngine:
    def decide_vote(self, citizen: Citizen, current_tick: int) -> dict: ...
    def decide_batch(self, citizens, current_tick: int):
        return [self.decide_vote(c, current_tick) for c in citizens]


class HeuristicLayaEngine(DecisionEngine):
    ECONOMIC_CONCERN_THRESHOLD = 55.0

    def _compat(self, citizen, cand):
        return sum(citizen.political_values[d] * cand.ideology_vector[d] for d in VALUES_DIMS) * 10.0

    def _group_bonus(self, citizen, cand):
        return 2.0 * len(set(citizen.groups) & set(cand.target_groups))

    def _scores(self, citizen, current_tick):
        mem = citizen.memory_by_government(current_tick)
        scores, detail = {}, {}
        for cid, cand in CANDIDATES.items():
            memory_term = mem.get("government_" + cid[-1], 0.0)
            wellbeing_term = sum(citizen.wellbeing.values()) / max(1, len(citizen.wellbeing)) * 0.05
            incumbent_bonus = 1.5 if citizen.incumbent == cid and memory_term > 0 else 0.0
            penalty = cand.incumbency_penalty if citizen.incumbent == cid and citizen.wants_change_hint else 0.0
            s = (memory_term
                 + wellbeing_term
                 + self._compat(citizen, cand)
                 + self._group_bonus(citizen, cand)
                 + cand.promise_reliability * 2.0
                 + incumbent_bonus
                 - penalty
                 + float(rng.normal(0, 1.0)))       # ruído controlado
            scores[cid] = s
            detail[cid] = {"memoria": round(memory_term, 2), "ideologia": round(self._compat(citizen, cand), 2)}
        return scores, detail

    def decide_vote(self, citizen: Citizen, current_tick: int) -> dict:
        scores, detail = self._scores(citizen, current_tick)
        ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
        best, second = ranked[0], ranked[1]
        margin = best[1] - second[1]

        # probabilidade de abstenção sobe com baixa confiança institucional
        abstain_p = max(0.0, (50 - citizen.institutional_trust) / 100.0) * (1 - citizen.political_engagement + 0.2)
        vote = "abstain" if rng.random() < abstain_p * 0.4 else best[0]

        confidence = min(0.97, 0.40 + margin / 20.0)   # margem maior => "modelo" mais confiante
        fallback_used = confidence < 0.55              # regra 15.4 do spec

        worst_dim = min(citizen.wellbeing, key=citizen.wellbeing.get)
        mem = citizen.memory_by_government(current_tick)
        inc_gov = "government_" + (citizen.incumbent[-1] if citizen.incumbent else "")

        return {
            "citizen_id": citizen.citizen_id,
            "values": {
                "vote": vote,
                "main_priority": WELLBEING_TO_PRIORITY.get(worst_dim, "jobs"),
                "trust_current_government": citizen.institutional_trust >= 50,
                "wants_change": bool(citizen.wants_change_hint),
                "economic_concern": int(round(4 * (100 - citizen.wellbeing.get("economic", 50)) / 100)),
                "safety_concern": int(round(4 * (100 - citizen.wellbeing.get("safety", 50)) / 100)),
                "punish_previous_government": mem.get(inc_gov, 0) < -2,
            },
            "answer_confidence": {"vote": round(confidence, 3)},
            "probabilities": {"vote": self._softmax(scores)},
            "fallback_used": fallback_used,
            "score_detail": detail,
            "model": "heuristic-standin (trocar por laya)",
        }

    @staticmethod
    def _softmax(scores, temp=6.0):
        exps = {k: math.exp(v / temp) for k, v in scores.items()}
        z = sum(exps.values())
        return {k: round(v / z, 3) for k, v in exps.items()}


def citizen(**kw):
    kw.setdefault("wants_change_hint", False)
    wh = kw.pop("wants_change_hint")
    c = Citizen(**kw)
    c.wants_change_hint = wh
    return c

In [5]:
# ============================================================# CÉLULA 6 — cenários de eleitores (da seção 15 do spec)
# ============================================================
T = 120  # tick atual = fim do 5º mandato

def mem(tick, gov, etype, impact, ew, pr=0.9, tags=None):
    return Memory(tick=tick, government_id=gov, event_type=etype, emotional_weight=ew,
                  personal_impact=impact, perceived_responsibility=pr, tags=tags or [])

SCENARIOS = {}

# 1 — Idosa de baixa renda, doença crônica, periferia (o exemplo da seção 15.1)
SCENARIOS["aposentada_periferia"] = citizen(
    citizen_id="cit_00417", age=63, region="periferia_metropolitana",
    income_bracket="low", employment_status="retired", health_condition="chronic",
    household_children=0, groups=["elderly", "low_income", "chronic"],
    political_values={"seguranca_ordem": 0.4, "igualdade": 0.8, "liberdade": 0.5, "crescimento": 0.3},
    institutional_trust=45, wellbeing={"economic": 41, "health": 36, "safety": 48,
                                       "education": 50, "freedom": 63, "environment": 47},
    incumbent="candidate_B",
    memory=[mem(20, "government_B", "health_improvement", 8, 1.1),
            mem(60, "government_B", "health_improvement", 7, 1.0),
            mem(96, "government_B", "economic_decline", -4, 1.4),
            mem(50, "government_A", "safety_improvement", 5, 1.0),
            mem(80, "government_A", "promise_broken", -6, 1.6),
            mem(30, "government_C", "economic_decline", -5, 1.3)],
)

# 2 — Comerciante de classe média na capital, vítima recente de crime
SCENARIOS["comerciante_capital"] = citizen(
    citizen_id="cit_00882", age=41, region="capital",
    income_bracket="middle", employment_status="business_owner", health_condition="healthy",
    household_children=1, groups=["business_owner", "middle_income", "urban_resident"],
    political_values={"seguranca_ordem": 0.85, "igualdade": 0.4, "liberdade": 0.6, "crescimento": 0.8},
    institutional_trust=55, wellbeing={"economic": 58, "health": 60, "safety": 22,
                                       "education": 55, "freedom": 65, "environment": 55},
    incumbent="candidate_B",
    memory=[mem(118, "government_B", "safety_decline", -9, 1.7),
            mem(90, "government_B", "economic_improvement", 4, 0.9),
            mem(70, "government_C", "economic_improvement", 6, 1.0)],
)

# 3 — Produtor rural que se beneficiou de obras de infraestrutura do candidato C
SCENARIOS["produtor_rural"] = citizen(
    citizen_id="cit_02201", age=52, region="interior_agricola",
    income_bracket="middle", employment_status="business_owner", health_condition="healthy",
    household_children=3, groups=["rural_producer", "business_owner"],
    political_values={"seguranca_ordem": 0.6, "igualdade": 0.35, "liberdade": 0.5, "crescimento": 0.9},
    institutional_trust=60, wellbeing={"economic": 70, "health": 55, "safety": 65,
                                       "education": 45, "freedom": 60, "environment": 40},
    incumbent="candidate_C",
    memory=[mem(100, "government_C", "economic_improvement", 8, 1.1),
            mem(110, "government_C", "promise_kept", 7, 1.2),
            mem(112, "government_C", "regional_neglect", -5, 1.4, tags=["environment"])],
    wants_change_hint=False,
)

# 4 — Estudante da capital, memória de escândalo de corrupção do governo C
SCENARIOS["estudante_desiludido"] = citizen(
    citizen_id="cit_03310", age=22, region="capital",
    income_bracket="low", employment_status="student", health_condition="healthy",
    household_children=0, groups=["student", "urban_resident", "minority_group"],
    political_values={"seguranca_ordem": 0.3, "igualdade": 0.75, "liberdade": 0.9, "crescimento": 0.5},
    institutional_trust=25, political_engagement=0.6,
    wellbeing={"economic": 35, "health": 60, "safety": 50,
               "education": 42, "freedom": 40, "environment": 50},
    incumbent="candidate_C",
    memory=[mem(105, "government_C", "corruption_scandal", -9, 1.8, pr=0.95),
            mem(100, "government_C", "freedom_decline", -7, 1.6),
            mem(60, "government_A", "freedom_decline", -6, 1.5),
            mem(40, "government_B", "education_improvement", 6, 1.0)],
    wants_change_hint=True,
)

# 5 — Pai de família da periferia: creche abriu, mas violência piorou
SCENARIOS["pai_periferia"] = citizen(
    citizen_id="cit_04711", age=34, region="periferia_metropolitana",
    income_bracket="low", employment_status="informal", health_condition="healthy",
    household_children=2, groups=["family_with_children", "low_income", "worker", "urban_resident"],
    political_values={"seguranca_ordem": 0.7, "igualdade": 0.7, "liberdade": 0.4, "crescimento": 0.5},
    institutional_trust=40, wellbeing={"economic": 45, "health": 55, "safety": 30,
                                       "education": 60, "freedom": 55, "environment": 45},
    incumbent="candidate_B",
    memory=[mem(115, "government_B", "education_improvement", 9, 1.2, pr=0.9),
            mem(96, "government_B", "safety_decline", -8, 1.6),
            mem(70, "government_A", "safety_improvement", 6, 1.1)],
)

# 6 — Empresário de alta renda: impostos subiram no governo B
SCENARIOS["empresario_alta_renda"] = citizen(
    citizen_id="cit_05001", age=48, region="capital",
    income_bracket="high", employment_status="business_owner", health_condition="healthy",
    household_children=2, groups=["high_income", "business_owner"],
    political_values={"seguranca_ordem": 0.7, "igualdade": 0.2, "liberdade": 0.6, "crescimento": 0.95},
    institutional_trust=65, wellbeing={"economic": 62, "health": 75, "safety": 60,
                                       "education": 70, "freedom": 70, "environment": 60},
    incumbent="candidate_B",
    memory=[mem(80, "government_B", "economic_decline", -7, 1.5),
            mem(50, "government_C", "economic_improvement", 8, 1.1),
            mem(100, "government_B", "promise_broken", -5, 1.5)],
    wants_change_hint=True,
)

## Cenários individuais
Seis perfis inspirados nas bases eleitorais das seções 9–11 do spec:
idosa crônica da periferia, comerciante da capital, produtor rural, estudante desiludido,
pai de família da periferia e empresário de alta renda.

In [6]:
# ============================================================# CÉLULA 7 — rodar cenários e exibir decisões
# ============================================================
engine = HeuristicLayaEngine()
rows = []
for name, cz in SCENARIOS.items():
    d = engine.decide_vote(cz, T)
    rows.append({
        "cenario": name, "idade": cz.age, "regiao": cz.region,
        "voto": d["values"]["vote"],
        "prioridade": d["values"]["main_priority"],
        "preoc_econ": d["values"]["economic_concern"],
        "preoc_seg": d["values"]["safety_concern"],
        "pune_governo": d["values"]["punish_previous_government"],
        "confianca": d["answer_confidence"]["vote"],
        "fallback": d["fallback_used"],
        "mem_A": d["score_detail"]["candidate_A"]["memoria"],
        "mem_B": d["score_detail"]["candidate_B"]["memoria"],
        "mem_C": d["score_detail"]["candidate_C"]["memoria"],
    })

import pandas as pd
df_cenarios = pd.DataFrame(rows)
df_cenarios

,cenario,idade,regiao,voto,prioridade,preoc_econ,preoc_seg,pune_governo,confianca,fallback,mem_A,mem_B,mem_C
0,aposentada_periferia,63,periferia_metropolitana,candidate_B,health,2,2,False,0.654,False,-2.13,-0.97,-0.43
1,comerciante_capital,41,capital,candidate_C,safety,2,3,True,0.590,False,0.00,-11.63,1.27
2,produtor_rural,52,interior_agricola,candidate_C,environment,1,1,False,0.966,False,0.00,0.00,5.11
3,estudante_desiludido,22,capital,candidate_B,cost_of_living,3,2,True,0.819,False,-1.43,0.54,-15.64
4,pai_periferia,34,periferia_metropolitana,candidate_B,safety,2,3,False,0.804,False,1.40,2.65,0.00
5,empresario_alta_renda,48,capital,candidate_C,safety,2,2,True,0.608,False,0.00,-6.76,1.05


In [7]:
# ============================================================# CÉLULA 8 — inspecionar um cidadão em detalhe (JSON do contrato 19.1)
# ============================================================
detalhe = engine.decide_vote(SCENARIOS["estudante_desiludido"], T)
print(json.dumps(detalhe, indent=2, ensure_ascii=False, default=str))

{
  "citizen_id": "cit_03310",
  "values": {
    "vote": "abstain",
    "main_priority": "cost_of_living",
    "trust_current_government": false,
    "wants_change": true,
    "economic_concern": 3,
    "safety_concern": 2,
    "punish_previous_government": true
  },
  "answer_confidence": {
    "vote": 0.77
  },
  "probabilities": {
    "vote": {
      "candidate_A": 0.217,
      "candidate_B": 0.744,
      "candidate_C": 0.039
    }
  },
  "fallback_used": false,
  "score_detail": {
    "candidate_A": {
      "memoria": -1.43,
      "ideologia": 10.18
    },
    "candidate_B": {
      "memoria": 0.54,
      "ideologia": 15.62
    },
    "candidate_C": {
      "memoria": -15.64,
      "ideologia": 13.3
    }
  },
  "model": "heuristic-standin (trocar por laya)"
}


## Eleição em lote (500 cidadãos)
4 regiões com perfis do spec (seção 5.3). Governo incumbente = **candidate_B**
(Bem-Estar Coletivo) no fim do 5º mandato (tick 120). Memórias dos eleitores refletem
mandato de B: melhora em saúde/educação vs. piora no custo de vida.

In [8]:
# ============================================================# CÉLULA 9 — eleição em lote: 500 cidadãos sintéticos, 4 regiões
# ============================================================
REGIONS = {
    "capital":                    {"violence": 60, "health": 65, "safety": 50, "economic": 60},
    "periferia_metropolitana":    {"violence": 82, "health": 38, "safety": 30, "economic": 40},
    "interior_agricola":          {"violence": 35, "health": 50, "safety": 65, "economic": 55},
    "cidade_costeira_turistica":  {"violence": 45, "health": 55, "safety": 60, "economic": 58},
}
GROUP_POOL = ["low_income", "worker", "family_with_children", "elderly", "student",
              "business_owner", "rural_producer", "urban_resident", "minority_group"]

def make_citizen(i, region, incumbent):
    prof = REGIONS[region]
    income = rng.choice(["low", "middle", "high"], p=[0.45, 0.4, 0.15])
    groups = list(rng.choice(GROUP_POOL, size=int(rng.integers(1, 3)), replace=False))
    if income == "high": groups.append("high_income")
    if region == "interior_agricola": groups.append("rural_producer")
    inc_gov = "government_" + incumbent[-1]
    memories = []
    if rng.random() < 0.55:   # experiência com governo atual (B: saúde/educa melhoram, custo sobe)
        good = rng.random() < 0.5
        memories.append(mem(int(rng.integers(96, 120)), inc_gov,
                            "health_improvement" if good else "economic_decline",
                            float(rng.uniform(4, 9) if good else -rng.uniform(3, 8)),
                            float(rng.uniform(0.9, 1.2) if good else rng.uniform(1.3, 1.8))))
    if rng.random() < 0.2:    # memórias antigas de outros governos
        g = "government_" + rng.choice(["A", "C"])
        memories.append(mem(int(rng.integers(20, 80)), g,
                            rng.choice(["economic_improvement", "safety_decline", "corruption_scandal"]),
                            float(rng.uniform(-8, 8)), float(rng.uniform(0.9, 1.6))))
    cz = citizen(
        citizen_id=f"cit_{i:05d}", age=int(rng.integers(18, 80)), region=region,
        income_bracket=str(income),
        employment_status=str(rng.choice(["employed", "unemployed", "informal", "retired", "student"])),
        health_condition=str(rng.choice(["healthy", "chronic"], p=[0.8, 0.2])),
        household_children=int(rng.integers(0, 4)), groups=groups,
        political_values={d: float(rng.random()) for d in VALUES_DIMS},
        institutional_trust=float(rng.normal(45, 15).clip(5, 95)),
        political_engagement=float(rng.normal(0.8, 0.15).clip(0.1, 1.0)),
        wellbeing={"economic": float(rng.normal(prof["economic"], 12).clip(0, 100)),
                   "health": float(rng.normal(prof["health"], 12).clip(0, 100)),
                   "safety": float(rng.normal(prof["safety"], 15).clip(0, 100)),
                   "education": float(rng.normal(50, 15).clip(0, 100)),
                   "freedom": float(rng.normal(60, 10).clip(0, 100)),
                   "environment": float(rng.normal(50, 12).clip(0, 100))},
        memory=memories, incumbent=incumbent,
        wants_change_hint=bool(rng.random() < 0.3),
    )
    return cz

INCUMBENT = "candidate_B"
pop = []
pop += [make_citizen(i, "capital", INCUMBENT) for i in range(0, 175)]
pop += [make_citizen(i, "periferia_metropolitana", INCUMBENT) for i in range(175, 300)]
pop += [make_citizen(i, "interior_agricola", INCUMBENT) for i in range(300, 425)]
pop += [make_citizen(i, "cidade_costeira_turistica", INCUMBENT) for i in range(425, 500)]

AttributeError: 'float' object has no attribute 'clip'

In [ ]:
# ============================================================# CÉLULA 10 — urna
# ============================================================
decisions = engine.decide_batch(pop, T)

tally = pd.DataFrame([{"citizen_id": c.citizen_id, "region": c.region,
                       "income": c.income_bracket, "groups": ",".join(c.groups),
                       "vote": d["values"]["vote"],
                       "confidence": d["answer_confidence"]["vote"],
                       "fallback": d["fallback_used"],
                       "priority": d["values"]["main_priority"]}
                      for c, d in zip(pop, decisions)])

print("=== Resultado geral ===")
print(tally["vote"].value_counts())
print()
print("=== Votos por região ===")
print(pd.crosstab(tally["region"], tally["vote"]))
print()
print("=== Votos por faixa de renda ===")
print(pd.crosstab(tally["income"], tally["vote"]))
print()
print(f"Confiança média: {tally['confidence'].mean():.3f} | "
      f"Fallback (conf < 0.55): {tally['fallback'].mean()*100:.1f}% | "
      f"Abstenção: {(tally['vote']=='abstain').mean()*100:.1f}%")
print()
print("=== Prioridade dominante do eleitorado ===")
print(tally["priority"].value_counts())

## Como trocar pelo Laya real

```python
# pip install laya
from laya import Laya  # nome simbólico — ajustar à API real do pacote

SCHEMA_VOTO = { ... }  # da seção 15.2 do spec

class RealLayaEngine(DecisionEngine):
    def __init__(self, schema=SCHEMA_VOTO, min_confidence=0.55):
        self.model = Laya.load("laya-multilingual")   # checkpoint multilíngue
        self.schema = schema
        self.min_confidence = min_confidence
        self.fallback = HeuristicLayaEngine()          # reusa a heurística

    def decide_vote(self, citizen: Citizen, current_tick: int) -> dict:
        state = build_citizen_state(citizen, current_tick)  # JSON da seção 15.1
        out = self.model.predict_structured(state, self.schema, return_details=True)
        if out.answer_confidence["vote"] < self.min_confidence:   # regra 15.4
            fb = self.fallback.decide_vote(citizen, current_tick)
            fb["fallback_used"] = True
            return fb
        return { ... }   # mapear para o contrato 19.1
```

Próximos passos sugeridos: rotular decisões esperadas dos 6 cenários → dataset de fine-tune
do checkpoint Laya (o uso zero-shot tem acurácia fraca sem treino no domínio).